# Practice 03 — Classic ML

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/theDocWho/lbv-notebooks/blob/main/ai-ml/03-classic-ml.ipynb) [![Open in Kaggle](https://img.shields.io/badge/Kaggle-open-20BEFF?logo=kaggle&logoColor=white)](https://kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/theDocWho/lbv-notebooks/main/ai-ml/03-classic-ml.ipynb)

Companion drills for **Phase 3** of [Learn by Visualization](https://learn-by-visuallization.org/illustrated/index.html) —
pairs with [confusion matrix](https://learn-by-visuallization.org/illustrated/3-classic-ml/confusion-matrix.html),
[cross-validation](https://learn-by-visuallization.org/illustrated/3-classic-ml/cross-validation.html),
[bias–variance](https://learn-by-visuallization.org/illustrated/3-classic-ml/bias-variance.html) and the scrub-through
[k-means explainer](https://learn-by-visuallization.org/illustrated/3-classic-ml/kmeans-steps.html).

**How this works:** each exercise is a `# TODO` scaffold followed by a self-check cell. Fill in the TODO, re-run both cells, and turn the ❌ into a ✅. No answers are hidden in the notebook — the checks themselves tell you what "correct" means. Runs on local Jupyter, Google Colab, or Kaggle (free, no setup).

In [ ]:
NEEDED = [("numpy", "numpy"), ("sklearn", "scikit-learn")]

# ---- setup: run me first ----
import importlib.util, subprocess, sys
for pkg, pipname in NEEDED:
    if importlib.util.find_spec(pkg) is None:
        print(f"installing {pipname} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pipname])

RESULTS = {}
def check(name, fn, hint=""):
    """Self-check: PASS/FAIL with a hint. Never raises -- rerun the cell after editing your answer."""
    try:
        ok = bool(fn())
    except Exception as e:
        ok, hint = False, f"{hint} (your code raised {type(e).__name__}: {e})"
    RESULTS[name] = ok
    print(("\u2705 PASS" if ok else "\u274c FAIL") + f" \u2014 {name}" + ("" if ok else f"\n   hint: {hint}"))

import numpy as np
rng = np.random.default_rng(0)


### Exercise 1 — a first honest model
Split `X, y` 75/25 (with `random_state=0`), fit `LogisticRegression`, and store the **test**
accuracy in `acc`. Evaluating on training data is the cardinal sin — the split is the whole point.

*Hint: `train_test_split(X, y, random_state=0)`; `model.score(X_test, y_test)`.*

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = make_classification(n_samples=400, n_features=8, n_informative=5, random_state=0)

acc = None  # TODO: TEST accuracy of a LogisticRegression


In [ ]:
check("acc is a sensible test accuracy (0.7 < acc <= 1.0)",
      lambda: acc is not None and 0.7 < float(acc) <= 1.0,
      "fit on the train split, score on the test split")

### Exercise 2 — precision & recall by hand
From raw prediction arrays, compute `precision` and `recall` for the positive class using
**pure NumPy** (count TP, FP, FN yourself). Then the check compares against sklearn — if they
disagree, you've just found the difference between the two metrics the hard (best) way.

*Hint: TP = `((y_true == 1) & (y_pred == 1)).sum()`; precision = TP/(TP+FP), recall = TP/(TP+FN).*

In [ ]:
y_true = np.array([1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0])
y_pred = np.array([1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0])

precision = None  # TODO
recall    = None  # TODO


In [ ]:
from sklearn.metrics import precision_score, recall_score
check("precision matches sklearn",
      lambda: precision is not None and abs(precision - precision_score(y_true, y_pred)) < 1e-9,
      "precision = TP / (TP + FP): of what you FLAGGED, how much was real?")
check("recall matches sklearn",
      lambda: recall is not None and abs(recall - recall_score(y_true, y_pred)) < 1e-9,
      "recall = TP / (TP + FN): of what was REAL, how much did you catch?")

### Exercise 3 — k-means from scratch (the two-beat loop)
Implement `kmeans(P, k, iters=20)`: init centroids as the first k points, then repeat
**assign** (nearest centroid) / **update** (mean of members). Return `(centroids, labels)`.
The check demands your inertia lands within 5% of sklearn's on the same init.

*Hint: distances via broadcasting — `((P[:, None, :] - C[None, :, :])**2).sum(-1)` then `argmin(axis=1)`.
This is exactly the loop you scrubbed in the explainer.*

In [ ]:
from sklearn.datasets import make_blobs
P, _ = make_blobs(n_samples=300, centers=4, cluster_std=1.2, random_state=7)

def kmeans(P, k, iters=20):
    C = P[:k].copy()
    labels = np.zeros(len(P), dtype=int)
    # TODO: alternate assign / update `iters` times
    return C, labels


In [ ]:
def _inertia(P, C, labels):
    return sum(((P[labels == i] - C[i]) ** 2).sum() for i in range(len(C)))

from sklearn.cluster import KMeans
_sk = KMeans(n_clusters=4, init=P[:4], n_init=1, max_iter=50, random_state=0).fit(P)

check("your inertia is within 5% of sklearn's (same init)",
      lambda: _inertia(P, *kmeans(P, 4)) <= 1.05 * _sk.inertia_,
      "assign: argmin of squared distances; update: mean of each cluster's members")

### Exercise 4 — cross-validation, not one lucky split
Compute `cv_mean`: the mean 5-fold cross-validation accuracy of `LogisticRegression` on the
Exercise-1 data. One split is an anecdote; five folds is evidence.

*Hint: `cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=5).mean()`.*

In [ ]:
from sklearn.model_selection import cross_val_score

cv_mean = None  # TODO: mean of 5-fold CV accuracy


In [ ]:
check("cv_mean is a plausible mean accuracy (0.7 < m < 1.0)",
      lambda: cv_mean is not None and 0.7 < float(cv_mean) < 1.0,
      "cross_val_score(...).mean()")

### Exercise 5 — watch a tree overfit
Fit two `DecisionTreeClassifier`s (random_state=0): `deep` with no depth limit and `shallow`
with `max_depth=3`. Store each model's train and test accuracy in the four variables.
The check verifies the classic signature: the deep tree memorizes (train ≈ 1.0) but the
**gap** between its train and test accuracy is larger than the shallow tree's gap.

*Hint: reuse X_train/X_test from Exercise 1's split (recreate it if needed).*

In [ ]:
from sklearn.tree import DecisionTreeClassifier

Xtr, Xte, ytr, yte = train_test_split(X, y, random_state=0)

deep_train = deep_test = shallow_train = shallow_test = None  # TODO


In [ ]:
check("deep tree memorizes the training set (train acc = 1.0)",
      lambda: deep_train is not None and float(deep_train) == 1.0,
      "an unlimited tree can carve one leaf per sample")
check("deep tree's train-test gap exceeds the shallow tree's",
      lambda: None not in (deep_train, deep_test, shallow_train, shallow_test) and
              (deep_train - deep_test) > (shallow_train - shallow_test),
      "that widening gap IS overfitting -- the bias-variance explainer, in numbers")

### Stretch goal — ROC by hand
For the Exercise-1 model, get `predict_proba` scores, sweep thresholds from 0 to 1, and plot
TPR vs FPR. Compare with `sklearn.metrics.roc_curve`, then revisit the
[ROC & AUC explainer](https://learn-by-visuallization.org/illustrated/3-classic-ml/roc-auc.html).

In [ ]:
# stretch — your playground


In [ ]:
# ---- self-score ----
passed, total = sum(RESULTS.values()), len(RESULTS)
print(f"Self-score: {passed}/{total} checks passing")
print("\U0001f389 All green \u2014 move on to the next explainer!" if total and passed == total
      else "Rerun each check cell as you fill in the TODOs above.")


<!--APPLY_SECTION-->
---
## 🧩 Apply — practice problems  🔒
You've learned the ideas above. Now prove it. Implement each function and run its
`grade(...)` — the tests are **hidden**, so a ✅ means it's genuinely correct. Stuck? call
`hint('pN')`. Full solutions are **locked in the Appendix** at the very bottom.

In [ ]:
# --- Apply autograder (hidden tests) ---
import base64 as _b64, marshal as _msh
def _eq(a,b,tol=1e-6):
    if isinstance(b,(list,tuple)): return isinstance(a,(list,tuple)) and len(a)==len(b) and all(_eq(x,y,tol) for x,y in zip(a,b))
    if isinstance(b,dict): return isinstance(a,dict) and a.keys()==b.keys() and all(_eq(a[k],b[k],tol) for k in b)
    if isinstance(b,bool): return a is b or a==b
    if isinstance(b,float):
        try: return abs(a-b)<=tol
        except Exception: return False
    return a==b
_TESTS={"p1": "WwMAAAApAqkCWwMAAADpAQAAAOkAAAAAcgEAAABbAwAAAHIBAAAAcgIAAAByAgAAAGdVVVVVVVXlPykCqQJbAgAAAHICAAAAcgIAAABbAgAAAHICAAAAcgIAAABnAAAAAAAA8D8pAqkCWwQAAAByAQAAAHIBAAAAcgEAAAByAQAAAFsEAAAAcgIAAAByAQAAAHIBAAAAcgIAAABnAAAAAAAA4D8=", "p2": "WwMAAAApAqkC6WQAAADnmpmZmZmZyT8pAulQAAAA6RQAAAApAqkC6QoAAADnAAAAAAAA0D8pAukIAAAA6QIAAAApAqkC6QcAAADnAAAAAAAA4D8pAukDAAAA6QQAAAA="}
_HINTS={"p1": ["Count matches with zip, divide by len."], "p2": ["n_test = int(round(n*frac)); n_train = n - n_test."]}
_SOLUTIONS={}
def grade(pid, fn):
    cs=_msh.loads(_b64.b64decode(_TESTS[pid]))
    for k,(args,exp) in enumerate(cs,1):
        try: got=fn(*args)
        except Exception as e:
            print(f"❌ {pid}: hidden case {k}/{len(cs)} raised {type(e).__name__}: {e}. Try hint('{pid}')."); return False
        if not _eq(got,exp):
            print(f"❌ {pid}: hidden case {k}/{len(cs)} failed — not right yet. Try hint('{pid}')."); return False
    print(f'✅ {pid}: all {len(cs)} hidden tests passed 🎉'); return True
def hint(pid,n=None):
    hs=_HINTS.get(pid,[]); rng=range(len(hs)) if n is None else [n-1]
    [print(f'Hint {i+1}: {hs[i]}') for i in rng]
def reveal(pid):
    if pid not in _SOLUTIONS: print('🔒 Locked — run the Appendix cell at the bottom first.'); return
    print(_b64.b64decode(_SOLUTIONS[pid]).decode())
print('Apply autograder ready ✓')


### p1 · `accuracy(y_true, y_pred)`
Fraction of predictions that match the truth (float).

In [ ]:
def accuracy(y_true, y_pred):
    # TODO
    pass

grade('p1', accuracy)
# hint('p1')


### p2 · `split_sizes(n, frac)`
Return (n_train, n_test) for a test fraction `frac` (round test, train gets the rest).

In [ ]:
def split_sizes(n, frac):
    # TODO: test=round(n*frac); train=n-test
    pass

grade('p2', split_sizes)
# hint('p2')


### ✅ Score

In [ ]:
import io, contextlib
_pairs=[('p1', accuracy), ('p2', split_sizes)]
done=0
for pid, fn in _pairs:
    buf=io.StringIO()
    with contextlib.redirect_stdout(buf): done+=grade(pid, fn)
print(f'\nApply score: {done} / 2' + ('  — nailed it 🏆' if done==2 else ''))


<!--APPLY_SECTION-->
### Appendix — Solutions (locked 🔒)
Try first. Run this to unlock, then `reveal('pN')`.

In [ ]:
_SOLUTIONS={"p1": "ZGVmIGFjY3VyYWN5KHlfdHJ1ZSwgeV9wcmVkKToKICAgIHJldHVybiBzdW0oMSBmb3IgYSxiIGluIHppcCh5X3RydWUseV9wcmVkKSBpZiBhPT1iKS9sZW4oeV90cnVlKQ==", "p2": "ZGVmIHNwbGl0X3NpemVzKG4sIGZyYWMpOgogICAgdGU9aW50KHJvdW5kKG4qZnJhYykpCiAgICByZXR1cm4gKG4tdGUsIHRlKQ=="}
print('🔓 Unlocked. reveal("p1") / reveal("p2").')
